In [0]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Soma total de vendas por dia , ordens com status de 'Delivered' e estate = NY           #              #    Origem camada silver: customer                                                         #
#                          orders                                                           # 
#                          product                                                          #
#    Destino camada gold: sales_ny                                                          #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela silver                              #
# 2. Criar uma tabela gold com os dados da tabela silver                                    #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#

In [0]:
import sys
import importlib

# 1. Garante que alterações no arquivo .py sejam carregadas automaticamente
%load_ext autoreload
%autoreload 2

# 2. Adiciona a pasta pai ('Codigo') com o prefixo '/Workspace'
caminho_codigo = '/Workspace/Users/elisangela.soares.santos83@gmail.com/Projeto_Azure_Databricks/Codigo'
if caminho_codigo not in sys.path:
    sys.path.append(caminho_codigo)

# 3. Importa e força o recarregamento do módulo
import funcoes_diversas.funcoes_util
importlib.reload(funcoes_diversas.funcoes_util)

# 4. Importa as funções desejadas
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria



In [0]:
mapa_criado = gerar_tabela_temporaria(["orders"],"silver")

In [0]:
df_sales_ny_gold = spark.sql("""
    select ORD.shipped_date,
           round(sum(ORD.total_sale),2) as total_sale        
    from tmp_silver_orders as ORD
    where 1 = 1
      and ORD.state = 'NY'
      and ORD.status_descricao = 'Delivered'
      group by ORD.shipped_date     
""")


In [0]:
gold_path = buscar_caminho("gold")
# transformando consulta em uma tabela Silver 
df_sales_ny_gold.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{gold_path}/sales_ny")